# Train 5 binary classifiers (LOCAL — multi-seed, ngưỡng 0.5)

In [1]:
import sys, os, json
from pathlib import Path
ROOT = next((c for c in [Path.cwd(), *Path.cwd().parents] if (c / "src").is_dir() and (c / "data").is_dir()), Path.cwd())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.training.train_model import load_yaml_config, resolve_runtime_config, run
import pandas as pd
RAW = load_yaml_config(Path("config/train.yml"))
TASKS = ["env", "soc", "gov", "commitment", "specificity"]
OUTDIR = {"env": "topic_e", "soc": "topic_s", "gov": "topic_g", "commitment": "commitment", "specificity": "specificity"}
print("repo:", ROOT)

C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo: d:\esg-washing


## Train + eval 5 model

In [2]:
# Train 5 model: nạp best_params (nếu có) -> override config -> train MULTI-SEED (ngưỡng 0.5) + save.
def _mean(test, k):   # schema multi-seed: test[k] = {mean, std}
    v = test.get(k)
    return v.get("mean") if isinstance(v, dict) else v

results = {}
for t in TASKS:
    cfg = resolve_runtime_config(RAW, task=t)
    bp_path = ROOT / f"outputs/models/{OUTDIR[t]}/best_params_{t}.json"
    if bp_path.exists():
        bp = json.load(open(bp_path))
        if "max_length" in bp:
            cfg["model"]["max_length"] = bp.pop("max_length")   # max_length ở model, không phải training
        cfg["training"].update(bp)
        print(f"[{t}] dùng best_params: {bp_path.name}")
    else:
        print(f"[{t}] CHƯA có best_params -> default HP (chạy 02-tune trước để tốt hơn)")
    metrics = run(cfg)   # run() = multi-seed (đọc training.seeds) -> mean±std @ ngưỡng 0.5 (cách C)
    results[t] = metrics
    print(f"[{t}] test macro_f1 = {_mean(metrics.get('test', {}), 'macro_f1'):.4f} "
          f"({metrics.get('n_seeds')} seed, ngưỡng 0.5)\n")

[env] dùng best_params: best_params_env.json
Loading labels...
Train: 1568, Val: 196
label
0    1058
1     510
Name: count, dtype: int64


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


count    1568.000000
mean       47.084184
std        24.942396
min         8.000000
25%        30.000000
50%        42.000000
75%        57.000000
max       233.000000
Name: sentence, dtype: float64

Training...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,F1 Positive
1,0.300300,0.322758,0.907927,0.918367,0.876923
2,0.192900,0.222962,0.924898,0.933673,0.899225
3,0.133400,0.258931,0.947586,0.954082,0.929134
4,0.090500,0.380195,0.923655,0.933673,0.896000
5,0.068900,0.310531,0.930398,0.938776,0.906250


Loading labels...
Train: 1568, Val: 196
label
0    1058
1     510
Name: count, dtype: int64


...

Loading labels...
Train: 841, Val: 149
label
0    505
1    336
Name: count, dtype: int64


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


count    841.000000
mean     130.555291
std       71.670922
min       22.000000
25%       90.000000
50%      115.000000
75%      152.000000
max      837.000000
Name: sentence, dtype: float64

Training...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,F1 Positive
1,0.641200,0.481879,0.807558,0.818792,0.761062
2,0.400000,0.359591,0.843726,0.852349,0.807018
3,0.336200,0.359612,0.825731,0.838926,0.777778
4,0.216600,0.320374,0.880389,0.885906,0.854701
5,0.165400,0.391189,0.864001,0.865772,0.848485
6,0.111100,0.470913,0.849256,0.859060,0.810811



[multi-seed] 5 seeds @ ngưỡng 0.5 | test macro_f1={'mean': 0.808, 'std': 0.0113} | f1_positive={'mean': 0.7475, 'std': 0.0202}
Saved metrics summary to: outputs\models\specificity\metrics_summary.json
[specificity] test macro_f1 = 0.8080 (5 seed, ngưỡng 0.5)



## Kết quả PhoBERT-v2

In [3]:
# Bảng PhoBERT-v2 multi-seed (mean±std) @ ngưỡng 0.5 (cách C).
def g(test, k, s="mean"):
    v = test.get(k, {})
    return round(v.get(s, float("nan")), 4) if isinstance(v, dict) else v

rows = []
for t in TASKS:
    m = results.get(t, {}); te = m.get("test", {})
    rows.append({"task": t,
                 "macro_f1": g(te, "macro_f1"), "macro_std": g(te, "macro_f1", "std"),
                 "f1_pos": g(te, "f1_positive"), "f1_pos_std": g(te, "f1_positive", "std"),
                 "n_seeds": m.get("n_seeds")})
pd.DataFrame(rows)

,task,macro_f1,macro_std,f1_pos,f1_pos_std,n_seeds
0,env,0.9540,0.0164,0.9385,0.0227,5
1,soc,0.9112,0.0120,0.8914,0.0153,5
2,gov,0.8008,0.0170,0.7168,0.0234,5
3,commitment,0.7887,0.0078,0.7288,0.0098,5
4,specificity,0.8080,0.0113,0.7475,0.0202,5


## Baseline (majority / TF-IDF+LR) — kỳ vọng PhoBERT > baseline

In [4]:
# Baseline: majority + TF-IDF + LogisticRegression (để so với PhoBERT)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

rows = []
for t in TASKS:
    tr = pd.read_parquet(ROOT / f"data/vi_gold/{t}/train.parquet")
    te = pd.read_parquet(ROOT / f"data/vi_gold/{t}/test.parquet")
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    Xtr, Xte = vec.fit_transform(tr["sentence"]), vec.transform(te["sentence"])
    lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, tr["label"])
    maj = DummyClassifier(strategy="most_frequent").fit(Xtr, tr["label"])
    rows.append({"task": t,
                 "majority_f1": round(f1_score(te["label"], maj.predict(Xte), average="macro"), 3),
                 "tfidf_lr_f1": round(f1_score(te["label"], lr.predict(Xte), average="macro"), 3)})
pd.DataFrame(rows)

,task,majority_f1,tfidf_lr_f1
0,env,0.402,0.865
1,soc,0.376,0.844
2,gov,0.423,0.759
3,commitment,0.410,0.739
4,specificity,0.381,0.735
